# SMS Spam Detection — CodSoft ML Internship Task 4

**Objective:** Build a traditional Machine Learning classification system to predict whether an SMS message is Spam or Ham (not spam).

**Approach:** TF-IDF text vectorization with Naive Bayes, Logistic Regression, and Linear SVM classifiers.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('Libraries imported successfully.')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/spam.csv', encoding='latin-1')
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Dataset Inspection

In [ ]:
print('Column Names:', list(df.columns))
print()
print('Data Types:')
print(df.dtypes)
print()
print('Dataset Info:')
df.info()

In [ ]:
# Check missing values
print('Missing Values:')
print(df.isnull().sum())
print()
print('Non-null counts for unnamed columns:')
for col in ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']:
    print(f'  {col}: {df[col].notna().sum()} non-null out of {len(df)}')

In [ ]:
# Check duplicates
print(f'Total duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate rows (v1 + v2 only): {df.duplicated(subset=["v1", "v2"]).sum()}')

In [ ]:
# Target distribution
print('Target (v1) distribution:')
print(df['v1'].value_counts())
print()
print('Target (v1) proportions:')
print(df['v1'].value_counts(normalize=True).round(4))

## 4. Data Cleaning

**Observations:**
- Columns `Unnamed: 2`, `Unnamed: 3`, `Unnamed: 4` are mostly empty (only 50, 12, and 6 non-null values respectively). These are artifacts of the CSV format and will be dropped.
- Column `v1` is the target (label: ham/spam), `v2` is the SMS message text.
- 403 duplicate rows exist and will be removed.
- No missing values in `v1` or `v2`.

In [ ]:
# Drop unnecessary unnamed columns
cols_to_drop = [col for col in df.columns if col.startswith('Unnamed')]
df = df.drop(columns=cols_to_drop)
print(f'Dropped columns: {cols_to_drop}')

# Rename columns
df = df.rename(columns={'v1': 'label', 'v2': 'message'})
print(f'Columns after rename: {list(df.columns)}')

# Remove duplicates
n_before = len(df)
df = df.drop_duplicates(subset=['label', 'message'], keep='first').reset_index(drop=True)
n_after = len(df)
print(f'Removed {n_before - n_after} duplicate rows. Dataset: {n_before} -> {n_after} rows.')

df.head()

## 5. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
colors = ['#22c55e', '#ef4444']
counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Message Count by Class', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(counts.items()):
    axes[0].text(i, count + 30, str(count), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors,
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Ham: {counts["ham"]} ({counts["ham"]/len(df)*100:.1f}%)')
print(f'Spam: {counts["spam"]} ({counts["spam"]/len(df)*100:.1f}%)')

## 6. Message Length Analysis

In [ ]:
df['message_length'] = df['message'].apply(len)

print('Message length statistics by class:')
print(df.groupby('label')['message_length'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
for label, color in zip(['ham', 'spam'], colors):
    subset = df[df['label'] == label]
    axes[0].hist(subset['message_length'], bins=50, alpha=0.7, label=label, color=color, edgecolor='white')
axes[0].set_title('Message Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Message Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box plot
sns.boxplot(data=df, x='label', y='message_length', palette={'ham': '#22c55e', 'spam': '#ef4444'}, ax=axes[1])
axes[1].set_title('Message Length by Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Message Length (characters)')

plt.tight_layout()
plt.show()

## 7. Word Count Analysis

In [ ]:
df['word_count'] = df['message'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='label', y='word_count', palette={'ham': '#22c55e', 'spam': '#ef4444'}, ax=ax)
ax.set_title('Word Count by Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Label')
ax.set_ylabel('Word Count')
plt.tight_layout()
plt.show()

print('Word count statistics by class:')
print(df.groupby('label')['word_count'].describe().round(1))

## 8. Text Preprocessing

Lightweight preprocessing:
- Convert to string
- Lowercase
- Remove non-printable characters
- Normalize whitespace

We keep it simple because TF-IDF handles stop words and the SMS text is short.

In [ ]:
def clean_text(text):
    """Lightweight text cleaning for SMS messages."""
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r'[^\x20-\x7E]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['message_clean'] = df['message'].apply(clean_text)

# Encode labels
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

print('Sample cleaned messages:')
for i in range(5):
    print(f'  [{df["label"].iloc[i]}] {df["message_clean"].iloc[i][:80]}...')

## 9. Train-Test Split

In [ ]:
X = df['message_clean']
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')
print(f'\nTraining class distribution:')
print(y_train.value_counts())
print(f'\nTest class distribution:')
print(y_test.value_counts())

## 10. TF-IDF Feature Extraction

We use TF-IDF with:
- `max_features=5000` — reasonable vocabulary size for SMS
- `ngram_range=(1, 2)` — unigrams and bigrams to capture phrases like "free call"
- `stop_words='english'` — remove common English stop words
- `min_df=2` — ignore terms appearing in fewer than 2 documents

TF-IDF is fitted **only on training data** to avoid data leakage.

In [ ]:
tfidf_params = {
    'max_features': 5000,
    'ngram_range': (1, 2),
    'stop_words': 'english',
    'min_df': 2
}

# Quick preview of TF-IDF features
tfidf_preview = TfidfVectorizer(**tfidf_params)
X_train_tfidf = tfidf_preview.fit_transform(X_train)

print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf_preview.vocabulary_)}')
print(f'\nTop 20 features by mean TF-IDF score:')

feature_names = tfidf_preview.get_feature_names_out()
mean_tfidf = X_train_tfidf.mean(axis=0).A1
top_indices = mean_tfidf.argsort()[-20:][::-1]
for idx in top_indices:
    print(f'  {feature_names[idx]}: {mean_tfidf[idx]:.4f}')

## 11. Model Training & Comparison

In [ ]:
# Define pipelines (TF-IDF + Classifier together)
pipelines = {
    'Multinomial Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(**tfidf_params)),
        ('classifier', MultinomialNB())
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(**tfidf_params)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(**tfidf_params)),
        ('classifier', CalibratedClassifierCV(
            LinearSVC(max_iter=2000, random_state=42, dual='auto'), cv=3
        ))
    ]),
}

# Train and evaluate
results = {}

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
    }
    results[name] = metrics
    
    print(f'\n{name}:')
    for metric, value in metrics.items():
        print(f'  {metric}: {value:.4f}')

print('\nAll models trained successfully.')

## 12. Model Comparison

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.round(4)
print('Model Comparison:')
print(comparison_df.to_string())

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
bar_colors = ['#3b82f6', '#22c55e', '#f59e0b']

for i, metric in enumerate(metrics_list):
    values = [results[name][metric] for name in results]
    bars = axes[i].bar(list(results.keys()), values, color=bar_colors, edgecolor='white', linewidth=0.5)
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_ylim(0.6, 1.05)
    axes[i].tick_params(axis='x', rotation=20, labelsize=9)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Baseline Model Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 13. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, pipeline) in enumerate(pipelines.items()):
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Ham', 'Spam'])
    disp.plot(ax=axes[i], cmap='Blues', values_format='d')
    axes[i].set_title(name, fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 14. Detailed Classification Reports

In [ ]:
for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test)
    print(f'\n{"=" * 50}')
    print(f'{name}')
    print(f'{"=" * 50}')
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

## 15. Hyperparameter Tuning

Based on the baseline comparison, the best model is selected for tuning.
We use `GridSearchCV` with a small parameter grid to avoid unnecessary computation.

In [ ]:
# Identify the best baseline model
best_name = max(results, key=lambda k: results[k]['F1 Score'])
best_f1 = results[best_name]['F1 Score']
print(f'Best baseline model: {best_name} (F1: {best_f1:.4f})')
print(f'\nTuning {best_name}...')

# Build tuning pipeline based on best model
if best_name == 'Multinomial Naive Bayes':
    tuning_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(**tfidf_params)),
        ('classifier', MultinomialNB())
    ])
    param_grid = {
        'classifier__alpha': [0.1, 0.5, 1.0, 2.0],
        'tfidf__max_features': [3000, 5000, 7000],
    }
elif best_name == 'Logistic Regression':
    tuning_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(**tfidf_params)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    param_grid = {
        'classifier__C': [0.1, 1.0, 10.0],
        'tfidf__max_features': [3000, 5000, 7000],
    }
else:  # Linear SVM
    tuning_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(**tfidf_params)),
        ('classifier', CalibratedClassifierCV(
            LinearSVC(max_iter=2000, random_state=42, dual='auto'), cv=3
        ))
    ])
    param_grid = {
        'classifier__estimator__C': [0.1, 1.0, 10.0],
        'tfidf__max_features': [3000, 5000, 7000],
    }

search = GridSearchCV(
    tuning_pipeline, param_grid,
    cv=3, scoring='f1', n_jobs=-1, verbose=0
)
search.fit(X_train, y_train)

print(f'\nBest parameters: {search.best_params_}')
print(f'Best CV F1 Score: {search.best_score_:.4f}')

# Evaluate tuned model
tuned_model = search.best_estimator_
y_tuned_pred = tuned_model.predict(X_test)
tuned_metrics = {
    'Accuracy': accuracy_score(y_test, y_tuned_pred),
    'Precision': precision_score(y_test, y_tuned_pred, zero_division=0),
    'Recall': recall_score(y_test, y_tuned_pred),
    'F1 Score': f1_score(y_test, y_tuned_pred),
}

print(f'\nTuned {best_name} on test set:')
for metric, value in tuned_metrics.items():
    print(f'  {metric}: {value:.4f}')

## 16. Final Model Selection

In [ ]:
if tuned_metrics['F1 Score'] >= best_f1:
    print(f'Tuned {best_name} is selected as the final model.')
    final_model = tuned_model
    final_metrics = tuned_metrics
else:
    print(f'Baseline {best_name} is selected as the final model.')
    final_model = pipelines[best_name]
    final_metrics = results[best_name]

print(f'\nFinal Model Metrics:')
for metric, value in final_metrics.items():
    print(f'  {metric}: {value:.4f}')

# Final classification report
y_final_pred = final_model.predict(X_test)
print(f'\nFinal Classification Report:')
print(classification_report(y_test, y_final_pred, target_names=['Ham', 'Spam']))

## 17. Final Model — Confusion Matrix

In [ ]:
y_final_pred = final_model.predict(X_test)
cm = confusion_matrix(y_test, y_final_pred)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=['Ham', 'Spam'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Final Model — Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 18. Save Final Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
model_path = '../models/spam_sms_model.pkl'
joblib.dump(final_model, model_path)
print(f'Final model (TF-IDF + Classifier pipeline) saved to: {model_path}')

## 19. Quick Prediction Test

In [ ]:
# Load and test the saved model
loaded_model = joblib.load('../models/spam_sms_model.pkl')

test_messages = [
    'Hey, are you coming to the party tonight?',
    'WINNER! You have been selected for a free prize. Call now!',
    'Reminder: Your appointment is at 3pm tomorrow.',
    'FREE entry to win 1000 cash! Text WIN to 80085 now!',
    'Can you pick up milk on your way home?',
]

def clean_text_simple(text):
    text = text.lower()
    text = re.sub(r'[^\x20-\x7E]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for msg in test_messages:
    cleaned = clean_text_simple(msg)
    pred = loaded_model.predict([cleaned])[0]
    label = 'SPAM' if pred == 1 else 'HAM'
    
    prob_str = ''
    if hasattr(loaded_model, 'predict_proba'):
        proba = loaded_model.predict_proba([cleaned])[0]
        prob_str = f' (spam prob: {proba[1]:.4f})'
    
    print(f'[{label}]{prob_str} — {msg}')

## Summary

- **Dataset:** SMS Spam Collection — 5,169 unique messages after removing 403 duplicates
- **Class distribution:** 87.4% Ham, 12.6% Spam (imbalanced)
- **Feature extraction:** TF-IDF with unigrams + bigrams, max 5,000 features
- **Models trained:** Multinomial Naive Bayes, Logistic Regression, Linear SVM
- **Best model:** Selected based on F1-Score (most appropriate metric for imbalanced classification)
- **Final pipeline saved:** `models/spam_sms_model.pkl` (includes TF-IDF + Classifier)